# V2_T – Create test items (run in your test tenant before production)

> **AS-IS - independent personal/community project.**
> These notebooks are provided **AS-IS**, without warranty of any kind, express or implied.
> They are a personal/community contribution, not a Microsoft product or service, and are not published on
> behalf of, endorsed, supported or warranted by Microsoft. They do not represent or bind Microsoft and
> create no obligation or commitment by Microsoft. Product names, APIs and documentation are technical
> references only. Use at your own risk and validate in your own environment before production or deletion.

Creates one test item of each type the v2 notebooks cover, with a little data, in **one test workspace**. Run the
whole chain V2_0 to V2_6 against that workspace in your test tenant before touching production: same tenant
settings, capacity, Key Vault, ADLS, Azure DevOps and ADX as the real run.

| Item | Name | Test content |
|---|---|---|
| Lakehouse | `bkp_test_lakehouse` | `Files/raw/` (CSV, nested text file), Delta table `bkp_test_sales` (1,000 rows), a shortcut `Files/bkp_test_shortcut` (the backup must skip it) |
| SQLEndpoint | created with the Lakehouse | view `dbo.bkp_test_view` |
| Warehouse | `bkp_test_warehouse` | table `dbo.bkp_test_orders` (500 rows), a view and a stored procedure |
| Eventhouse + KQL database | `bkp_test_eventhouse` | table `bkp_test_events` (1,000 rows) and a function |
| Notebook | `bkp_test_notebook` | two cells, default Lakehouse = the test Lakehouse (V2_5 lists it under "References to deleted items") |
| DataPipeline | `bkp_test_datapipeline` | a Notebook activity that runs the test notebook |
| Dataflow | `bkp_test_dataflow` | one Power Query query (empty dataflow if the definition is rejected) |
| DataAgent, GraphModel, Ontology | `bkp_test_dataagent`, `bkp_test_graphmodel`, `bkp_test_ontology` | empty items (preview types; not available in every tenant) |

### How to use
1. Create a **test workspace on a Fabric capacity** (F SKU or trial) with a clear name, for example
   `migration-test-<date>`. Import this notebook into it (Language: **Python**, not PySpark). Leave
   `TARGET_WORKSPACE_ID` empty to use the notebook's own workspace.
2. **Run all.** About 5 to 15 minutes, mostly waiting for the Warehouse, the SQL endpoint and the Eventhouse.
   Running it again doesn't create duplicates or add rows twice.
3. The last cell prints the `WORKSPACE_IDS` line for the V2 notebooks. Then run, in order: V2_0, V2_1 (links the
   workspace to Git), V2_2, V2_3, V2_4 (dry run, then the real hard delete), V2_5, V2_6. Check that V2_6 ends
   with "ALL DATA VERIFIED".
4. Clean up **before the ATM deadlines**: every Fabric item in the tenant, test items included, must be
   hard-deleted (a soft-deleted item blocks the migration). Set `DELETE_TEST_ITEMS = True` in the last section and
   run that cell: it hard-deletes every item whose name starts with `bkp_test_`, plus the Dataflow Gen2 staging
   items of this workspace when `DELETE_DATAFLOW_STAGING = True`. Keep this notebook itself until the end, or delete
   it with Delete_Notebooks_By_Pattern.

Each step is independent: if one item type isn't available in your tenant, it's reported and the rest continue.
Don't run it in a production workspace: the cleanup deletes by name prefix.

In [ ]:
TARGET_WORKSPACE_ID = ""   # empty = the workspace this notebook is in
PREFIX = "bkp_test"        # every test item name starts with this
CREATE_TYPES = ["Lakehouse", "Warehouse", "Eventhouse", "Notebook", "DataPipeline", "Dataflow",
                "DataAgent", "GraphModel", "Ontology"]   # SQLEndpoint and KQLDatabase come with their parents
LARGE_DATA_MIB = 0         # 0 = original small fixtures; >0 = extra owned stress table bytes per storage family
MEASURE_DATA_SIZE = False  # True = also measure persisted sizes in small mode; never accept unpublished/zero sizes
ISOLATED_STRESS_RUNTIME = False  # True = use a hash-pinned Delta wheel in this GB seed job only; needs PyArrow 16+

## Helpers

In [ ]:
BUILD_ID = "ce3952e6bd1e"
print("V2_T_Create_Test_Items build ce3952e6bd1e (compare with BUILD_IDS.txt of the package you meant to run)")
# AS-IS, without warranty of any kind; independent personal/community code.
# Not a Microsoft product or service; not published on behalf of Microsoft.
# Does not represent or bind Microsoft; no Microsoft endorsement, warranty or support commitment.
"""Credential redaction for diagnostic output, never for API or backup payloads."""
import re
from collections.abc import Mapping
from typing import TypeVar, overload
from urllib.parse import quote

REDACTED = "<redacted>"
if "_REDACTION_VALUES" not in globals():
    _REDACTION_VALUES: set[str] = set()

_REDACTION_KEYS = {
    "authorization", "accesstoken", "refreshtoken", "clientsecret", "serviceprincipalsecret",
    "password", "passwd", "accountkey", "apikey", "personalaccesstoken", "pat", "secret", "token",
    "connectionstring", "sastoken", "sharedaccesssignature", "sig",
}
_REDACTION_PATTERNS = (
    re.compile(r"\bgh[pousr]_[A-Za-z0-9]{20,}\b"),
    re.compile(r"\bgithub_pat_[A-Za-z0-9_]{40,}\b"),
    re.compile(r"\beyJ[A-Za-z0-9_-]{16,}\.[A-Za-z0-9_-]{16,}\.[A-Za-z0-9_-]{16,}\b"),
    re.compile(r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----.*?"
               r"-----END (?:RSA |EC |OPENSSH )?PRIVATE KEY-----", re.DOTALL),
)
_REDACTION_AUTH = re.compile(r"\b(Bearer|Basic)([ \t]+)[A-Za-z0-9_+/=.-]{16,}", re.IGNORECASE)
_REDACTION_QUERY = re.compile(
    r"([?&](?:sig|access_token|refresh_token|client_secret|password|api_key)=)[^&\s\"'<>]+",
    re.IGNORECASE,
)
_REDACTION_ASSIGNMENT = re.compile(
    r"""(\b(?:client[_-]?secret|servicePrincipalSecret|access[_-]?token|refresh[_-]?token|
    password|passwd|account[_-]?key|api[_-]?key|personalAccessToken|sas[_-]?token|pat|secret|token)
    \b["']?\s*[:=]\s*["']?)([^\s"'&,;}\]<>]+)""",
    re.IGNORECASE | re.VERBOSE,
)


def register_sensitive_value(value: str) -> None:
    if len(value) >= 8:
        _REDACTION_VALUES.add(value)


def redact_sensitive_text(text: str) -> str:
    for value in sorted(_REDACTION_VALUES.copy(), key=len, reverse=True):
        text = text.replace(value, REDACTED).replace(quote(value, safe=""), REDACTED)
    for pattern in _REDACTION_PATTERNS:
        text = pattern.sub(REDACTED, text)
    text = _REDACTION_AUTH.sub(lambda m: m[1] + m[2] + REDACTED, text)
    text = _REDACTION_QUERY.sub(lambda m: m[1] + REDACTED, text)
    return _REDACTION_ASSIGNMENT.sub(lambda m: m[1] + REDACTED, text)


_Key = TypeVar("_Key")


@overload
def redact_sensitive_data(value: str) -> str: ...


@overload
def redact_sensitive_data(value: Mapping[_Key, object]) -> dict[_Key, object]: ...


@overload
def redact_sensitive_data(value: list[object]) -> list[object]: ...


@overload
def redact_sensitive_data(value: tuple[object, ...]) -> tuple[object, ...]: ...


@overload
def redact_sensitive_data(value: object) -> object: ...


def redact_sensitive_data(value: object) -> object:
    if isinstance(value, Mapping):
        out = {}
        for key, item in value.items():
            normalized = re.sub(r"[^a-z0-9]", "", str(key).lower())
            if normalized in _REDACTION_KEYS and item is not None and not (isinstance(item, str) and item == ""):
                out[key] = REDACTED
            else:
                out[key] = redact_sensitive_data(item)
        return out
    if isinstance(value, list):
        return [redact_sensitive_data(item) for item in value]
    if isinstance(value, tuple):
        return tuple(redact_sensitive_data(item) for item in value)
    return redact_sensitive_text(value) if isinstance(value, str) else value
import base64, json, re, time, uuid
import pandas as pd
import requests

try:
    notebookutils  # preloaded in Fabric notebooks
except NameError:
    import notebookutils

FABRIC_API = "https://api.fabric.microsoft.com/v1"
ONELAKE_DFS = "https://onelake.dfs.fabric.microsoft.com"
STEPS = []
_session = requests.Session()


class HttpError(Exception):
    def __init__(self, status, code, message, url=""):
        self.status, self.code, self.message, self.url = status, code or "", redact_sensitive_text(str(message or "")), url
        super().__init__(f"HTTP {status} {code}: {self.message}" if status else f"{code}: {self.message}")


def _details(resp):
    try:
        body = resp.json()
    except ValueError:
        return "", resp.text or ""
    err = body.get("error") if isinstance(body, dict) and isinstance(body.get("error"), dict) else body
    if not isinstance(err, dict):
        return "", str(err)
    return str(err.get("errorCode") or err.get("code") or ""), str(err.get("message") or "")


def http(method, url, audience, *, params=None, json_body=None, data=None, headers=None, ok404=False):
    for attempt in range(1, 7):
        hdrs = {"Authorization": "Bearer " + notebookutils.credentials.getToken(audience), **(headers or {})}
        register_sensitive_value(hdrs["Authorization"].split(" ", 1)[-1])
        try:
            resp = _session.request(method, url, params=params, json=json_body, data=data, headers=hdrs, timeout=300)
        except requests.RequestException as exc:
            if attempt == 6:
                raise HttpError(None, type(exc).__name__, str(exc), url)
            time.sleep(5 * attempt)
            continue
        if resp.status_code in (429, 408) or resp.status_code >= 500:
            if attempt < 6:
                try:
                    wait = int(float(resp.headers.get("Retry-After", "")))
                except ValueError:
                    wait = 10 * attempt
                time.sleep(min(max(wait, 1), 300))
                continue
        if ok404 and resp.status_code == 404:
            return None
        if resp.status_code >= 400:
            raise HttpError(resp.status_code, *_details(resp), url)
        return resp


def fabric(method, path, body=None):
    resp = http(method, FABRIC_API + path, "pbi", json_body=body)
    if resp.status_code == 202 and (resp.headers.get("Location") or resp.headers.get("x-ms-operation-id")):
        location = resp.headers.get("Location") or f"{FABRIC_API}/operations/{resp.headers['x-ms-operation-id']}"
        deadline = time.monotonic() + 30 * 60
        while True:
            time.sleep(3)
            state = http("GET", location, "pbi").json()
            status = str(state.get("status", "")).lower()
            if status == "succeeded":
                result = http("GET", location.rstrip("/") + "/result", "pbi", ok404=True)
                return result.json() if result is not None and result.content else {}
            if status in ("failed", "cancelled", "canceled"):
                err = state.get("error") or {}
                raise HttpError(None, err.get("errorCode") or status, err.get("message"), location)
            if time.monotonic() > deadline:
                raise HttpError(None, "Timeout", "operation didn't finish in 30 minutes", location)
    return resp.json() if resp.content else {}


def fabric_list(path):
    url, out = FABRIC_API + path, []
    while url:
        body = http("GET", url, "pbi").json()
        out += body.get("value") or []
        url = body.get("continuationUri")
    return out


def b64(data):
    if not isinstance(data, (bytes, bytearray)):
        data = data if isinstance(data, str) else json.dumps(data, indent=2)
        data = data.encode("utf-8")
    return base64.b64encode(data).decode()


def find_item(item_type, name):
    for item in fabric_list(f"/workspaces/{WS}/items?type={item_type}"):
        if item.get("displayName", "").lower() == name.lower() and item.get("type") == item_type:
            return item
    return None


def ensure_item(item_type, name, definition=None, description="Test item for the Fabric migration notebooks (V2_T)"):
    """Return (item id, 'Exists' or 'Created'). Creates the item only if no item of that type has this name."""
    found = find_item(item_type, name)
    if found:
        if (LARGE_DATA_MIB or MEASURE_DATA_SIZE) and item_type in ("Lakehouse", "Warehouse", "Eventhouse"):
            if found.get("description") != description:
                raise RuntimeError(f"Ownership collision: {item_type} {name} isn't a V2_T-owned test item. "
                                   "Use a fresh PREFIX; don't adopt or overwrite existing user data.")
        return found["id"], "Exists"
    body = {"displayName": name, "type": item_type, "description": description}
    if definition:
        body["definition"] = definition
    result = fabric("POST", f"/workspaces/{WS}/items", body)
    item_id = result.get("id") if isinstance(result, dict) else None
    if not item_id:
        found = find_item(item_type, name)
        item_id = found["id"] if found else None
    if not item_id:
        raise HttpError(None, "NotFound", f"{item_type} {name} was accepted but can't be found")
    return item_id, "Created"


def step(item_type, name, item_id, status, detail=""):
    STEPS.append({"Type": item_type, "Name": name, "Id": item_id or "", "Status": status, "Detail": str(detail)})
    print(f"  {item_type:<12} {name:<28} {status}  {detail}")


def wait_until(check, what, timeout=15 * 60, every=10):
    deadline = time.monotonic() + timeout
    while True:
        value = check()
        if value:
            return value
        if time.monotonic() > deadline:
            raise HttpError(None, "Timeout", f"{what} wasn't ready after {timeout // 60} minutes")
        time.sleep(every)


def onelake_write(item_id, path, data):
    if isinstance(data, str):
        data = data.encode("utf-8")
    url = f"{ONELAKE_DFS}/{WS}/{item_id}/{path}"
    hdr = {"x-ms-version": "2023-11-03"}
    http("PUT", url, "storage", params={"resource": "file"}, headers=hdr)
    if data:
        http("PATCH", url, "storage", params={"action": "append", "position": "0"}, data=data, headers=hdr)
    http("PATCH", url, "storage", params={"action": "flush", "position": str(len(data))}, data=b"", headers=hdr)


def sql_connect(item_id, item_type):
    data_utils = getattr(notebookutils, "data", None)
    if data_utils is None or not hasattr(data_utils, "connect_to_artifact"):
        raise RuntimeError("notebookutils.data isn't available: set the notebook language to Python, not PySpark")
    return data_utils.connect_to_artifact(item_id, WS, item_type)


def sql_run(conn, statement):
    """Run a T-SQL statement that returns no rows (wrapped in EXEC so it can be followed by a SELECT)."""
    escaped = statement.replace("'", "''")
    conn.query(f"EXEC(N'{escaped}'); SELECT 1 AS done;")
    if hasattr(conn, "commit"):
        try:
            conn.commit()
        except Exception:
            pass


def sql_scalar(conn, query):
    df = conn.query(query)
    return None if df is None or len(df) == 0 else df.iloc[0, 0]


def kusto(uri, database, command, kind="mgmt"):
    resp = http("POST", uri.rstrip("/") + f"/v1/rest/{kind}", "kusto", json_body={"db": database, "csl": command})
    body = resp.json()
    if body.get("error") or body.get("Exceptions"):
        raise HttpError(None, "KustoError", json.dumps(body))
    tables = body.get("Tables") or []
    if not tables:
        return []
    cols = [c["ColumnName"] for c in tables[0].get("Columns", [])]
    return [dict(zip(cols, row)) for row in tables[0].get("Rows", [])]


if type(LARGE_DATA_MIB) is not int or LARGE_DATA_MIB < 0:
    raise ValueError("LARGE_DATA_MIB must be a non-negative integer number of MiB (0 keeps the small fixtures).")
if type(MEASURE_DATA_SIZE) is not bool:
    raise ValueError("MEASURE_DATA_SIZE must be True or False.")
if type(ISOLATED_STRESS_RUNTIME) is not bool:
    raise ValueError("ISOLATED_STRESS_RUNTIME must be True or False.")
if (LARGE_DATA_MIB or MEASURE_DATA_SIZE) and not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{0,79}", PREFIX):
    raise ValueError("Measured/stress PREFIX must start with a letter and contain at most 80 letters, digits or _.")
if LARGE_DATA_MIB and "Lakehouse" not in CREATE_TYPES:
    raise ValueError("LARGE_DATA_MIB requires Lakehouse in CREATE_TYPES: its owned source is reused by the bulk loaders.")

ctx = notebookutils.runtime.context
WS = (TARGET_WORKSPACE_ID or ctx.get("currentWorkspaceId") or "").strip().lower()
if not WS:
    raise ValueError("Set TARGET_WORKSPACE_ID: the notebook's own workspace couldn't be read.")
ws_info = fabric("GET", f"/workspaces/{WS}")
print(f"Target workspace: {ws_info.get('displayName')} ({WS})")
if not ws_info.get("capacityId"):
    print("WARNING: this workspace has no capacity. Lakehouse, Warehouse, Eventhouse and most other items "
          "need a Fabric capacity (F SKU or trial).")
NAMES = {t: f"{PREFIX}_{t.lower()}" for t in CREATE_TYPES}
IDS = {}

In [ ]:
# Optional one-copy stress producer. No Spark session, external credentials, or package installation.
import hashlib, io, math
from urllib.parse import quote, unquote

_SEED_TARGET_BYTES = LARGE_DATA_MIB * 1024**2
_SEED_MAX_ROWS = 8192                # at most ~16 MiB of payload per in-memory frame
_SEED_MAX_BATCHES = 256             # shared source batches, including any consumer-specific top-ups
_SEED_PUBLISH_ATTEMPTS = 90          # 15 minutes, also bounded by the current family's overall deadline
_SEED_POLL_SECONDS = 10
_SEED_MAX_SECONDS = 2 * 60 * 60     # per storage family; no background work
_SEED_MAX_FILE_BYTES = 64 * 1024**2
_SEED_COLUMNS = [("seed_owner", "string"), ("seed_batch", "long"), ("seed_row", "long"), ("payload", "string")]
_SEED_DELTA_CACHE = {}
_SEED_POOL = None
_SEED_DEADLINE = None
SEED_METRICS = {}
if LARGE_DATA_MIB or MEASURE_DATA_SIZE:
    for _family in ("Lakehouse", "Warehouse", "Eventhouse"):
        if _family in CREATE_TYPES:
            SEED_METRICS[_family] = {
                "itemId": "", "table": f"{PREFIX}_stress" if LARGE_DATA_MIB else
                f"{PREFIX}_{dict(Lakehouse='sales', Warehouse='orders', Eventhouse='events')[_family]}",
                "rows": 0, "storedBytes": 0, "targetBytes": _SEED_TARGET_BYTES, "status": "PENDING",
                "sizeMetric": "compressed_extent_bytes" if _family == "Eventhouse" else "active_parquet_bytes"}


class SeedError(RuntimeError):
    pass


class _SeedNotReady(SeedError):
    pass


def _seed_check_deadline():
    if _SEED_DEADLINE is not None and time.monotonic() > _SEED_DEADLINE:
        raise SeedError("Stress producer time limit reached. Rerun the same parameters to resume owned batches; "
                        "inspect capacity/publishing/ingestion errors first. No data was removed.")


def _seed_relative(path):
    decoded = unquote(path)
    if (not decoded or decoded.startswith("/") or "\\" in decoded or ":" in decoded
            or any(p in ("", ".", "..") for p in decoded.split("/"))):
        raise SeedError(f"Unsafe OneLake/Delta path: {path!r}. Refusing an external or ambiguous data file.")
    return decoded


def _seed_url(item_id, path):
    return f"{ONELAKE_DFS}/{WS}/{item_id}/{quote(_seed_relative(path), safe='/')}"


def _seed_read(item_id, path):
    head = http("HEAD", _seed_url(item_id, path), "storage", ok404=True)
    if head is None:
        return None
    try:
        length = int(head.headers["Content-Length"])
    except (KeyError, ValueError) as exc:
        raise SeedError(f"Missing/invalid OneLake file length at {path}; refusing an unbounded download.") from exc
    if not 0 <= length <= _SEED_MAX_FILE_BYTES:
        raise SeedError(f"OneLake file at {path} exceeds the 64 MiB seed-reader memory bound; no data was downloaded.")
    return http("GET", _seed_url(item_id, path), "storage", ok404=True)


def _seed_list(item_id, directory):
    """List metadata only. In particular, never infer Warehouse bytes from a data-file listing."""
    out, continuation, seen = [], None, set()
    for _ in range(64):
        _seed_check_deadline()
        params = {"resource": "filesystem", "directory": f"{item_id}/{_seed_relative(directory)}",
                  "recursive": "false", "maxResults": "5000"}
        if continuation:
            params["continuation"] = continuation
        resp = http("GET", f"{ONELAKE_DFS}/{WS}", "storage", params=params, ok404=True)
        if resp is None:
            return out
        out.extend(resp.json().get("paths") or [])
        continuation = resp.headers.get("x-ms-continuation")
        if not continuation:
            return out
        if continuation in seen:
            raise SeedError("OneLake returned a repeated continuation token; refusing an incomplete size measurement.")
        seen.add(continuation)
    raise SeedError("OneLake metadata listing exceeded 64 pages; narrow the test scope instead of assuming a size.")


def _seed_stats(add):
    stats = add.get("stats")
    if isinstance(stats, str):
        stats = json.loads(stats)
    return stats if isinstance(stats, dict) else (add.get("stats_parsed") or {})


def _seed_checkpoint_rows(raw):
    import pyarrow.parquet as pq  # already part of the Fabric Python / deltalake runtime
    reader = pq.ParquetFile(io.BytesIO(raw))
    for batch in reader.iter_batches(batch_size=1024):
        yield from batch.to_pylist()


def seed_delta_snapshot(item_id, table_path):
    """Replay active Delta add/remove actions, including classic checkpoints. add.size is physical Parquet bytes.

    Warehouse data files can be invisible in REST listings. Only _delta_log is listed/read here.
    No row length, Parquet footer uncompressed size, or historical/tombstoned file is counted.
    """
    table_path = _seed_relative(table_path)
    entries = _seed_list(item_id, table_path + "/_delta_log")
    names = {p["name"].rsplit("/", 1)[-1] for p in entries if str(p.get("isDirectory", "")).lower() != "true"}
    logs = sorted(int(n[:20]) for n in names if re.fullmatch(r"\d{20}\.json", n))
    if not names:
        return None
    if len(logs) > 4096:
        raise SeedError("Delta log exceeds the bounded test-item reader (4096 commits). Use a fresh test PREFIX.")
    cached = _SEED_DELTA_CACHE.get((item_id, table_path))
    active, metadata, version = {}, None, -1

    def apply(action):
        nonlocal metadata
        if action.get("metaData") is not None:
            metadata = action["metaData"]
        if action.get("remove") is not None:
            active.pop(action["remove"]["path"], None)
        if action.get("add") is not None:
            add = action["add"]
            _seed_relative(add["path"])
            if not add["path"].lower().endswith(".parquet") or add.get("deletionVector"):
                raise SeedError("Unsupported active Delta file/deletion vector; cannot certify test-table stored size.")
            if type(add.get("size")) is not int or add["size"] < 0:
                raise SeedError("Delta add.size is missing/invalid; a logical row size is not a storage metric.")
            active[add["path"]] = add
            if len(active) > 16384:
                raise SeedError("Active Delta file count exceeds the seed reader's metadata memory bound (16384).")

    checkpoints = [int(n[:20]) for n in names if
                   re.fullmatch(r"\d{20}\.checkpoint(?:\.\d{10}\.\d{10})?\.parquet", n)]
    tail = [n for n in logs if cached and n > cached["version"]]
    cache_usable = (cached and logs and logs[-1] >= cached["version"]
                    and max(checkpoints, default=-1) <= logs[-1]
                    and (not tail or (tail[0] == cached["version"] + 1
                                      and all(b == a + 1 for a, b in zip(tail, tail[1:])))))
    full_log = logs and logs[0] == 0 and all(b == a + 1 for a, b in zip(logs, logs[1:]))
    if cache_usable:
        active, metadata, version = dict(cached["active"]), cached["metadata"], cached["version"]
    elif not full_log:
        last = _seed_read(item_id, table_path + "/_delta_log/_last_checkpoint")
        if last is None:
            raise _SeedNotReady("Delta log gap: version 0 and a complete checkpoint are not published yet.")
        cp = last.json()
        if "v2Checkpoint" in cp:
            raise SeedError("V2 Delta checkpoints are not supported by this bounded seed reader. "
                            "Use a fresh test table or add a verified reader; don't waive the size check.")
        version = int(cp["version"])
        parts = int(cp.get("parts") or 1)
        if not 1 <= parts <= 64:
            raise SeedError("Delta checkpoint exceeds the bounded seed reader (64 parts).")
        for i in range(1, parts + 1):
            name = (f"{version:020d}.checkpoint.parquet" if parts == 1 else
                    f"{version:020d}.checkpoint.{i:010d}.{parts:010d}.parquet")
            resp = _seed_read(item_id, table_path + "/_delta_log/" + name)
            if resp is None:
                raise _SeedNotReady(f"Delta checkpoint part not published: {name}")
            if len(resp.content) > _SEED_MAX_FILE_BYTES:
                raise SeedError("Delta checkpoint part exceeds the seed reader's 64 MiB memory bound.")
            for action in _seed_checkpoint_rows(resp.content):
                apply(action)
    for n in logs:
        if n <= version:
            continue
        if n != version + 1:
            raise _SeedNotReady(f"Delta log gap: expected {version + 1}, found {n}. Wait for log publication.")
        resp = _seed_read(item_id, f"{table_path}/_delta_log/{n:020d}.json")
        if resp is None:
            raise _SeedNotReady(f"Delta version {n} is not published yet.")
        if len(resp.content) > _SEED_MAX_FILE_BYTES:
            raise SeedError("Delta commit exceeds the seed reader's 64 MiB memory bound.")
        if not resp.content.strip():
            raise _SeedNotReady(f"Empty/incomplete published Delta commit {n}. Wait for commit publication.")
        try:
            for line in resp.content.decode("utf-8").splitlines():
                if line.strip():
                    apply(json.loads(line))
        except (ValueError, UnicodeError) as exc:
            raise _SeedNotReady(f"Incomplete/invalid published Delta commit {n}: {exc}") from exc
        version = n
    rows = 0
    for add in active.values():
        count = _seed_stats(add).get("numRecords")
        if type(count) is not int or count < 0:
            raise _SeedNotReady("Delta numRecords statistics not published; cannot align byte size with actual rows.")
        rows += count
    snap = {"active": active, "metadata": metadata, "version": version, "rows": rows,
            "storedBytes": sum(a["size"] for a in active.values())}
    _SEED_DELTA_CACHE[(item_id, table_path)] = snap
    return snap


def _seed_wait(check, what):
    last = "no published result"
    for attempt in range(_SEED_PUBLISH_ATTEMPTS):
        _seed_check_deadline()
        try:
            value = check()
            if value is not None:
                return value
        except _SeedNotReady as exc:
            last = str(exc)
        if attempt + 1 < _SEED_PUBLISH_ATTEMPTS:
            time.sleep(_SEED_POLL_SECONDS)
    raise SeedError(f"{what}: no valid positive persisted size after {_SEED_PUBLISH_ATTEMPTS} polls. {last} "
                    "Inspect Delta publishing/ingestion and existing permissions, then rerun the same parameters. "
                    "No roles, policies or data were changed to bypass this check.")


def seed_size_status(stored_bytes, target_bytes, rows):
    if type(rows) is not int or rows <= 0 or type(stored_bytes) is not int or stored_bytes <= 0:
        raise SeedError(f"Invalid persisted size measurement: rows={rows}, storedBytes={stored_bytes}. "
                        "A populated table cannot be reported as successfully measured at zero bytes.")
    if target_bytes and stored_bytes > target_bytes * 1.10:
        raise SeedError(f"Persisted size {stored_bytes} exceeds 110% of target {target_bytes}. "
                        "No cleanup/shrink is attempted; use a fresh PREFIX after investigating the physical metric.")
    return "OK" if not target_bytes or stored_bytes >= target_bytes * .95 else "GROWING"


def _seed_record(family, item_id, table, rows, stored_bytes, target_bytes):
    metric = {"itemId": item_id, "table": table, "rows": int(rows), "storedBytes": int(stored_bytes),
              "targetBytes": int(target_bytes), "sizeMetric": "compressed_extent_bytes" if family == "Eventhouse"
              else "active_parquet_bytes", "status": seed_size_status(int(stored_bytes), target_bytes, int(rows))}
    SEED_METRICS[family] = metric
    return metric


def seed_fail(family, exc):
    if family in SEED_METRICS:
        metric = SEED_METRICS[family]
        metric.update(status="Failed", error=str(exc))
        metric["itemId"] = metric["itemId"] or IDS.get(family, "")


def seed_measure_delta(family, item_id, table, table_path, expected_rows, target_bytes=0):
    def published():
        snap = seed_delta_snapshot(item_id, table_path)
        if snap is None or snap["rows"] != expected_rows or snap["storedBytes"] <= 0:
            raise _SeedNotReady(f"Delta publishing: expected {expected_rows} rows and positive active Parquet bytes; "
                                f"observed {None if snap is None else (snap['rows'], snap['storedBytes'])}.")
        return snap
    snap = _seed_wait(published, f"{family} {table}")
    return _seed_record(family, item_id, table, expected_rows, snap["storedBytes"], target_bytes)


def seed_measure_kql(item_id, uri, database, table, expected_rows, target_bytes=0):
    """CompressedSize of *all* current extents, not hot cache, OriginalSize or an estimate.

    Metric definition: https://learn.microsoft.com/kusto/management/show-extents
    Direct extents avoid .show table details' cached summary (documented as up to 15 minutes old).
    This excludes index bytes and replicas; it is the engine's compressed-extent byte metric, not billing size.
    """
    def published():
        extents = kusto(uri, database, f".show table {table} extents")
        rows, size = 0, 0
        for extent in extents:
            compressed, count = extent.get("CompressedSize"), extent.get("RowCount")
            if (isinstance(compressed, bool) or not isinstance(compressed, (int, float))
                    or not math.isfinite(compressed) or compressed < 0 or type(count) is not int or count < 0):
                raise _SeedNotReady("KQL CompressedSize/RowCount missing or invalid; OriginalSize isn't accepted.")
            if extent.get("DeletedRowCount", 0):
                raise SeedError("KQL extent has deleted rows; this is not an untouched owned seed table.")
            rows += count
            size += compressed
        if rows != expected_rows or rows <= 0 or size <= 0:
            raise _SeedNotReady(f"KQL publishing: expected {expected_rows} rows/positive compressed bytes; "
                                f"observed rows={rows}, compressedBytes={size}.")
        return int(size)
    size = _seed_wait(published, f"Eventhouse {table}")
    return _seed_record("Eventhouse", item_id, table, expected_rows, size, target_bytes)

In [ ]:
import importlib.metadata
import platform
import subprocess
import sys
import tempfile
from pathlib import Path

_SEED_DELTA_VERSION = "0.22.3"
_SEED_RUNTIME_TEMP = None
_SEED_DELTA_WHEELS = {
    "linux": (
        "https://files.pythonhosted.org/packages/7c/09/95abe066d746fb3be6c74dce78f3c847edf8b274062fb73b0b5afdae3d8f/"
        "deltalake-0.22.3-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",
        "636c657ac0a6f439c4f6a180f43f4d521e91ced8a8041cb69001de0e566518b9"),
    "win32": (
        "https://files.pythonhosted.org/packages/f6/c4/10f33c1b6aee253fe88ba4ad02d7ab1532b2de9f91a9df5b1e435e9fabdb/"
        "deltalake-0.22.3-cp39-abi3-win_amd64.whl",
        "65d9b477ff6e7ae2333da86d268551b3c7ff6654ba65f1c4900670f8ad6fba2b"),
}


def _seed_download_wheel(url, expected_sha):
    digest, data = hashlib.sha256(), io.BytesIO()
    # Public package download: never send notebook credentials to the package host.
    resp = requests.get(url, stream=True, timeout=300)
    try:
        resp.raise_for_status()
        for chunk in resp.iter_content(1024 * 1024):
            if data.tell() + len(chunk) > _SEED_MAX_FILE_BYTES:
                raise SeedError("Pinned Delta wheel exceeds the 64 MiB download bound.")
            digest.update(chunk)
            data.write(chunk)
    finally:
        resp.close()
    if digest.hexdigest() != expected_sha:
        raise SeedError("Pinned Delta wheel SHA-256 mismatch. No package was installed or imported.")
    return data.getvalue()


def seed_prepare_runtime():
    global _SEED_RUNTIME_TEMP
    if _SEED_RUNTIME_TEMP is not None:
        return
    if "deltalake" in sys.modules:
        raise SeedError("The kernel already imported deltalake. Start a fresh Python job for the isolated stress "
                        "runtime; no loaded native module will be replaced.")
    if sys.platform not in _SEED_DELTA_WHEELS or platform.machine().lower() not in ("amd64", "x86_64"):
        raise SeedError("The pinned stress runtime supports Linux/Windows x86-64 only; no unverified wheel was used.")
    try:
        arrow_version = importlib.metadata.version("pyarrow")
    except importlib.metadata.PackageNotFoundError as exc:
        raise SeedError("The isolated Delta runtime requires existing PyArrow 16+. No shared environment was changed.") from exc
    if int(arrow_version.split(".", 1)[0]) < 16:
        raise SeedError(f"The isolated Delta runtime requires existing PyArrow 16+, found {arrow_version}. "
                        "No loaded/shared PyArrow package was changed.")
    url, expected_sha = _SEED_DELTA_WHEELS[sys.platform]
    data = _seed_download_wheel(url, expected_sha)
    _SEED_RUNTIME_TEMP = tempfile.TemporaryDirectory(prefix="v2t-stress-delta-")
    root = Path(_SEED_RUNTIME_TEMP.name)
    wheel = root / url.rsplit("/", 1)[-1]
    wheel.write_bytes(data)
    target = root / "packages"
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--target", str(target), str(wheel)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=300)
    if result.returncode:
        raise SeedError(f"Job-local Delta installation failed (exit {result.returncode}):\n{result.stdout}")
    sys.path.insert(0, str(target))
    import deltalake
    if deltalake.__version__ != _SEED_DELTA_VERSION:
        raise SeedError(f"Wrong isolated Delta version: {deltalake.__version__}; expected {_SEED_DELTA_VERSION}.")
    print(f"Stress-only runtime: deltalake {_SEED_DELTA_VERSION}, SHA-256 verified, existing PyArrow {arrow_version}; "
          "temporary job directory only; no shared environment change.")


if ISOLATED_STRESS_RUNTIME and LARGE_DATA_MIB:
    seed_prepare_runtime()

In [ ]:
def seed_payload(owner, row_id):
    """2 KiB deterministic, independent high-entropy ASCII per row; retries regenerate identical data."""
    entropy = hashlib.shake_256(f"V2_T_STRESS_V1|{owner}|{row_id}".encode()).digest(1536)
    return base64.b64encode(entropy).decode("ascii")


def _seed_frame(plan):
    count, start = plan["rows"], plan["first_row"]
    if not 1 <= count <= _SEED_MAX_ROWS:
        raise SeedError("Source batch exceeds the bounded in-memory row limit.")
    values = [seed_payload(plan["seed_owner"], i) for i in range(start, start + count)]
    frame = pd.DataFrame({"seed_owner": [plan["seed_owner"]] * count, "seed_batch": [plan["seed_batch"]] * count,
                          "seed_row": range(start, start + count), "payload": values})
    digest = hashlib.sha256()
    for value in values:
        digest.update(value.encode("ascii"))
    if plan.get("payloadSha256") and plan["payloadSha256"] != digest.hexdigest():
        raise SeedError("Synthetic source fingerprint changed. Keep the same producer/runtime or use a fresh PREFIX.")
    return frame, digest.hexdigest()


def _seed_encode_parquet(frame):
    import pyarrow as pa
    import pyarrow.parquet as pq
    out = io.BytesIO()
    pq.write_table(pa.Table.from_pandas(frame, preserve_index=False), out, compression="snappy",
                   use_dictionary=False, row_group_size=min(len(frame), 1024))
    data = out.getvalue()
    if len(data) > _SEED_MAX_FILE_BYTES:
        raise SeedError("Generated source Parquet exceeds the 64 MiB per-file memory bound.")
    return data


def _seed_properties(resp):
    props = {}
    for entry in resp.headers.get("x-ms-properties", "").split(","):
        if "=" in entry:
            k, v = entry.strip().split("=", 1)
            props[k] = base64.b64decode(v).decode("utf-8")
    return props


def _seed_put_once(item_id, path, data, owner, intent=False):
    """Exclusive file create + verified prefix append. Never truncate even an interrupted owned upload.

    Small JSON intents are also in file properties so an interrupted intent itself can be repaired.
    Parquet bytes are deterministically regenerated from the preceding immutable intent.
    """
    _seed_check_deadline()
    if not isinstance(data, bytes) or len(data) > _SEED_MAX_FILE_BYTES:
        raise SeedError("Owned upload must be bytes within the 64 MiB memory bound.")
    sha = hashlib.sha256(data).hexdigest()
    props = {"v2tOwner": owner, "v2tSha256": sha}
    if intent:
        if len(data) > 4096:
            raise SeedError("Source JSON intent exceeds its 4 KiB metadata bound.")
        props["v2tIntent"] = data.decode("utf-8")
    hdr = {"x-ms-version": "2023-11-03", "If-None-Match": "*",
           "x-ms-properties": ",".join(k + "=" + b64(v) for k, v in props.items())}
    try:
        http("PUT", _seed_url(item_id, path), "storage", params={"resource": "file"}, headers=hdr)
    except HttpError as exc:
        if exc.status not in (409, 412):
            raise
    existing = _seed_read(item_id, path)
    if existing is None:
        raise SeedError(f"Owned file disappeared during upload: {path}. Rerun after investigating concurrent activity.")
    actual_props = _seed_properties(existing)
    if any(actual_props.get(k) != v for k, v in props.items()) or not data.startswith(existing.content):
        raise SeedError(f"Ownership/content collision at {path}. Use a fresh PREFIX; no existing data was overwritten.")
    if existing.content == data:
        return
    position = len(existing.content)
    etag = existing.headers.get("ETag")
    if not etag:
        raise SeedError(f"Missing OneLake ETag at {path}. Refusing to append without a conditional flush.")
    http("PATCH", _seed_url(item_id, path), "storage", params={"action": "append", "position": str(position)},
         data=data[position:], headers={"x-ms-version": "2023-11-03"})
    # ADLS rejects conditional headers on append; the ETag condition belongs on the committing flush.
    http("PATCH", _seed_url(item_id, path), "storage", params={"action": "flush", "position": str(len(data))},
         data=b"", headers={"x-ms-version": "2023-11-03", "If-Match": etag})
    verified = _seed_read(item_id, path)
    if verified is None or hashlib.sha256(verified.content).hexdigest() != sha:
        raise SeedError(f"Owned source checksum verification failed: {path}. Rerun the same parameters; don't overwrite it.")


def _seed_json_bytes(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8")


def _seed_get_plan(item_id, path, owner):
    resp = _seed_read(item_id, path)
    if resp is None:
        raise SeedError(f"Missing tracked source intent: {path}. Refusing to infer ownership from a filename.")
    props = _seed_properties(resp)
    if props.get("v2tOwner") != owner or not props.get("v2tIntent"):
        raise SeedError(f"Unowned source intent at {path}. Use a fresh PREFIX.")
    expected = props["v2tIntent"].encode("utf-8")
    if hashlib.sha256(expected).hexdigest() != props.get("v2tSha256"):
        raise SeedError(f"Corrupt source intent properties at {path}. No data was changed.")
    if resp.content != expected:
        _seed_put_once(item_id, path, expected, owner, intent=True)
    return json.loads(expected)


def _seed_lh_path(lh_id, table):
    props = fabric("GET", f"/workspaces/{WS}/lakehouses/{lh_id}").get("properties") or {}
    schema = props.get("defaultSchema") or ""
    if not isinstance(schema, str) or (schema and not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{0,127}", schema)):
        raise SeedError("Unknown Lakehouse schema layout. Inspect defaultSchema before creating owned test data.")
    return f"Tables/{schema + '/' if schema else ''}{table}"


def seed_small_lakehouse(lh_id, frame):
    """The measured/stress path never overwrites even the original small fixture. Legacy mode is unchanged."""
    table = f"{PREFIX}_sales"
    path = _seed_lh_path(lh_id, table)
    snap = seed_delta_snapshot(lh_id, path)
    description = f"V2_T_SMALL_V1:{WS}:{PREFIX}"
    if snap is not None:
        meta = snap["metadata"] or {}
        columns = [f["name"] for f in json.loads(meta.get("schemaString") or "{}").get("fields", [])]
        if meta.get("description") != description or columns != list(frame.columns) or snap["rows"] != len(frame):
            raise SeedError(f"Ownership/schema collision at {table}. Leave existing fixtures alone and use a fresh PREFIX.")
        return path
    if _seed_list(lh_id, path):
        raise SeedError(f"Non-Delta data already exists at {path}. Refusing to overwrite it.")
    from deltalake import write_deltalake
    write_deltalake(f"abfss://{WS}@{ONELAKE_DFS.split('://', 1)[1]}/{lh_id}/{path}", frame, mode="error",
                    description=description,
                    storage_options={"bearer_token": notebookutils.credentials.getToken("storage"),
                                     "use_fabric_endpoint": "true"})
    return path


def _seed_validate_plan(plan, pool, batch, start):
    if (plan.get("seed_owner") != pool["owner"] or plan.get("seed_batch") != batch or plan.get("first_row") != start
            or type(plan.get("rows")) is not int or not 1 <= plan["rows"] <= _SEED_MAX_ROWS
            or plan.get("kind") not in ("delta", "parquet")
            or not re.fullmatch(r"[0-9a-f]{64}", plan.get("payloadSha256", ""))):
        raise SeedError("Source ownership/schema/range collision in a tracked batch. Use a fresh PREFIX.")
    if plan["kind"] == "parquet":
        if (plan.get("path") != f"{pool['root']}/source/{batch:06d}.parquet"
                or not re.fullmatch(r"[0-9a-f]{64}", plan.get("fileSha256", ""))
                or type(plan.get("storedBytes")) is not int or not 0 < plan["storedBytes"] <= _SEED_MAX_FILE_BYTES):
            raise SeedError("Unsafe tracked Parquet source path/size/checksum.")


def _seed_pool():
    global _SEED_POOL
    if _SEED_POOL is not None:
        return _SEED_POOL
    lh_id = IDS.get("Lakehouse")
    if not lh_id:
        raise SeedError("Large seeding requires this group's successfully created Lakehouse.")
    table, root = f"{PREFIX}_stress", f"Files/_v2t_stress/{PREFIX}"
    path = _seed_lh_path(lh_id, table)
    scope = {"producer": "V2_T_STRESS_V1", "workspaceId": WS, "lakehouseId": lh_id, "prefix": PREFIX,
             "targetBytes": _SEED_TARGET_BYTES, "payloadBytes": 2048, "tablePath": path}
    owner = hashlib.sha256(_seed_json_bytes(scope)).hexdigest()
    pool = {"lh": lh_id, "table": table, "tablePath": path, "root": root, "owner": owner, "plans": [],
            "description": "V2_T_STRESS_V1:" + owner}
    snap = seed_delta_snapshot(lh_id, path)
    if snap is not None and (snap["metadata"] or {}).get("description") != pool["description"]:
        raise SeedError(f"Ownership collision at {path}. Never adopt an existing user/other-profile table.")
    if snap is None and _seed_list(lh_id, path):
        raise SeedError(f"Existing non-Delta/unpublished data at {path}. Refusing to overwrite or infer ownership.")
    _seed_put_once(lh_id, root + "/owner.json", _seed_json_bytes(scope), owner, intent=True)
    entries = _seed_list(lh_id, root + "/batches")
    files = sorted(p["name"].rsplit("/", 1)[-1] for p in entries)
    if len(files) > _SEED_MAX_BATCHES or any(not re.fullmatch(r"\d{6}\.json", f) for f in files):
        raise SeedError("Unexpected/too many source intents. Refusing untracked synthetic ranges.")
    start, raw_seen = 0, False
    for n, filename in enumerate(files):
        if filename != f"{n:06d}.json":
            raise SeedError("Gap in tracked source batches. Do not append past a missing intent.")
        plan = _seed_get_plan(lh_id, f"{root}/batches/{filename}", owner)
        _seed_validate_plan(plan, pool, n, start)
        if raw_seen and plan["kind"] == "delta":
            raise SeedError("Lakehouse append appears after frozen source top-ups. Refusing an ambiguous source history.")
        raw_seen |= plan["kind"] == "parquet"
        pool["plans"].append(plan)
        start += plan["rows"]
    _seed_lh_groups(pool, snap)
    _SEED_POOL = pool
    return pool


def _seed_lh_groups(pool, snap):
    if snap is None:
        return {}
    meta = snap["metadata"] or {}
    schema = json.loads(meta.get("schemaString") or "{}")
    columns = [(f.get("name"), f.get("type")) for f in schema.get("fields", [])]
    if meta.get("description") != pool["description"] or columns != _SEED_COLUMNS or meta.get("partitionColumns"):
        raise SeedError("Owned Lakehouse stress table ownership/schema changed. No append is safe.")
    groups = {}
    for filename, add in snap["active"].items():
        stats = _seed_stats(add)
        lo, hi, nulls = stats.get("minValues", {}), stats.get("maxValues", {}), stats.get("nullCount", {})
        batch, first, last = lo.get("seed_batch"), lo.get("seed_row"), hi.get("seed_row")
        if (type(batch) is not int or hi.get("seed_batch") != batch or type(first) is not int or type(last) is not int
                or lo.get("seed_owner") != pool["owner"] or hi.get("seed_owner") != pool["owner"]
                or any(nulls.get(c) != 0 for c, _ in _SEED_COLUMNS)
                or last - first + 1 != stats["numRecords"]):
            raise SeedError("Lakehouse source statistics don't describe a complete owned batch/range. "
                            "Inspect compaction/schema/concurrent writers before resuming.")
        groups.setdefault(batch, []).append((first, last, stats["numRecords"], filename, add["size"]))
    result = {}
    for batch, ranges in groups.items():
        ranges.sort()
        if not 0 <= batch < len(pool["plans"]):
            raise SeedError("Untracked batch in the owned Lakehouse. Refusing to append.")
        plan = pool["plans"][batch]
        pos = plan["first_row"]
        for first, last, count, _, _ in ranges:
            if first != pos:
                raise SeedError("Overlapping/missing source rows; refusing retry duplicates.")
            pos = last + 1
        if plan["kind"] != "delta" or pos != plan["first_row"] + plan["rows"]:
            raise SeedError("Published Lakehouse range doesn't match its immutable intent.")
        result[batch] = {"seed_batch": batch, "rows": plan["rows"], "first_row": plan["first_row"],
                         "paths": [(pool["tablePath"] + "/" + r[3], r[4]) for r in ranges]}
    if sorted(result) != list(range(len(result))):
        raise SeedError("Gap in committed Lakehouse batches; no append is safe.")
    return result


def _seed_delta_runtime():
    try:
        from deltalake import write_deltalake, CommitProperties, Transaction, WriterProperties
    except ImportError as exc:
        raise SeedError("Large seeding needs the existing Fabric Python deltalake runtime with CommitProperties, "
                        "Transaction and WriterProperties (delta-rs Python 0.22+ API). Use an approved compatible "
                        "Fabric Python environment; no packages are installed by this producer. " + str(exc)) from exc
    return write_deltalake, CommitProperties, Transaction, WriterProperties


def _seed_source(batch, wanted_rows, as_delta=False):
    """Reuse the same immutable source for WH and KQL. Only top-ups go to Files, not the size-gated LH table."""
    pool = _seed_pool()
    _seed_check_deadline()
    if not 0 <= batch < _SEED_MAX_BATCHES:
        raise SeedError(f"Source batch limit {_SEED_MAX_BATCHES} reached before the physical size target. "
                        "Inspect compression/metrics; no logical-size success fallback is allowed.")
    plans = pool["plans"]
    data, frame = None, None
    if batch == len(plans):
        if as_delta and any(p["kind"] != "delta" for p in plans):
            raise SeedError("Lakehouse size changed after consumer top-ups. Use a fresh PREFIX instead of enlarging it blindly.")
        plan = {"seed_owner": pool["owner"], "seed_batch": batch, "first_row": sum(p["rows"] for p in plans),
                "rows": int(wanted_rows), "kind": "delta" if as_delta else "parquet"}
        frame, plan["payloadSha256"] = _seed_frame(plan)
        if not as_delta:
            data = _seed_encode_parquet(frame)
            plan.update(path=f"{pool['root']}/source/{batch:06d}.parquet",
                        fileSha256=hashlib.sha256(data).hexdigest(), storedBytes=len(data))
        _seed_put_once(pool["lh"], f"{pool['root']}/batches/{batch:06d}.json", _seed_json_bytes(plan),
                       pool["owner"], intent=True)
        plans.append(plan)
    elif batch < len(plans):
        plan = plans[batch]
    else:
        raise SeedError("Source batch gap; refusing to generate an untracked range.")
    if plan["kind"] == "delta":
        snap = seed_delta_snapshot(pool["lh"], pool["tablePath"])
        groups = _seed_lh_groups(pool, snap)
        if batch not in groups:
            writer, CommitProperties, Transaction, WriterProperties = _seed_delta_runtime()
            if frame is None:
                frame, _ = _seed_frame(plan)
            writer(f"abfss://{WS}@{ONELAKE_DFS.split('://', 1)[1]}/{pool['lh']}/{pool['tablePath']}", frame,
                   mode="error" if snap is None else "append", description=pool["description"],
                   storage_options={"bearer_token": notebookutils.credentials.getToken("storage"),
                                    "use_fabric_endpoint": "true"},
                   writer_properties=WriterProperties(compression="SNAPPY", statistics_truncate_length=128),
                   # Delta 0.22 counts total commit attempts here; zero never attempts an append.
                   commit_properties=CommitProperties(app_transactions=[Transaction("v2t-" + pool["owner"], batch)],
                                                      max_commit_retries=1))
            def committed():
                current = seed_delta_snapshot(pool["lh"], pool["tablePath"])
                found = _seed_lh_groups(pool, current)
                if batch not in found:
                    raise _SeedNotReady(f"Owned Delta batch {batch} not published yet.")
                return found
            groups = _seed_wait(committed, "Owned Lakehouse source commit")
        return {**plan, **groups[batch]}
    if as_delta:
        raise SeedError("Cannot append a raw consumer source into the frozen size-gated Lakehouse table.")
    if data is None:
        frame, _ = _seed_frame(plan)
        data = _seed_encode_parquet(frame)
    if len(data) != plan["storedBytes"] or hashlib.sha256(data).hexdigest() != plan["fileSha256"]:
        raise SeedError("Parquet source encoding changed. Keep the original approved runtime; don't replace existing source files.")
    _seed_put_once(pool["lh"], plan["path"], data, pool["owner"])
    return {**plan, "paths": [(plan["path"], plan["storedBytes"])]}


def _seed_destination_marker(family, item_id, table, exists):
    pool = _seed_pool()
    path = f"{pool['root']}/{family.lower()}-{item_id}.json"
    value = {"producer": pool["description"], "itemId": item_id, "table": table,
             "columns": _SEED_COLUMNS, "sourceLakehouseId": pool["lh"]}
    if _seed_read(pool["lh"], path) is None and exists:
        raise SeedError(f"Ownership collision at {family} {table}: a table exists without its owned intent. "
                        "Use a fresh PREFIX; even an empty matching schema isn't permission to adopt user data.")
    _seed_put_once(pool["lh"], path, _seed_json_bytes(value), pool["owner"], intent=True)

In [ ]:
# One writer per (TARGET_WORKSPACE_ID, PREFIX). Parent owns copy count, distinct prefixes and serialization.
# Reruns reconcile committed ranges first; they never repeat a bulk load just because its acknowledgement was lost.

def _seed_validate_groups(records):
    pool, found = _seed_pool(), {}
    for row in records:
        batch = int(row["seed_batch"])
        if batch in found or not 0 <= batch < len(pool["plans"]):
            raise SeedError("Untracked/duplicate destination batch. Refusing to ingest into existing user data.")
        plan = pool["plans"][batch]
        if (int(row["rows"]) != plan["rows"] or int(row["distinct_rows"]) != plan["rows"]
                or int(row["first_row"]) != plan["first_row"]
                or int(row["last_row"]) != plan["first_row"] + plan["rows"] - 1
                or row["owner_min"] != pool["owner"] or row["owner_max"] != pool["owner"]
                or int(row["bad_payload"]) != 0):
            raise SeedError(f"Destination batch {batch} ownership/schema/range mismatch or partial ingestion. "
                            "No repair/duplicate append is attempted. Inspect the batch; use a fresh PREFIX if needed.")
        found[batch] = plan
    if sorted(found) != list(range(len(found))):
        raise SeedError("Destination has a batch gap. Refusing to append past unverified synthetic ranges.")
    return found


def _seed_verify_hashes(records, rows):
    pool = _seed_pool()
    expected = {i: hashlib.sha256(seed_payload(pool["owner"], i).encode()).hexdigest() for i in {0, rows - 1}}
    observed = {int(r["seed_row"]): str(r["payload_hash"]).lower() for r in records}
    if len(records) != len(expected) or observed != expected:
        raise SeedError("Destination first/last payload SHA-256 mismatch. Row counts alone aren't content verification.")


def _seed_drive(family, item_id, table, read_groups, measure, append, verify=None):
    global _SEED_DEADLINE
    _SEED_DEADLINE = time.monotonic() + _SEED_MAX_SECONDS
    for _ in range(_SEED_MAX_BATCHES + 1):
        _seed_check_deadline()
        groups = read_groups()
        rows = sum(p["rows"] for p in groups.values())
        stored = 0
        if rows:
            metric = measure(rows)
            stored = metric["storedBytes"]
            if metric["status"] == "OK":
                if verify:
                    verify(rows)
                plans = _seed_pool()["plans"][:len(groups)]
                metric["sourceFingerprint"] = hashlib.sha256(
                    "".join(f"{p['seed_batch']}:{p['rows']}:{p['payloadSha256']};" for p in plans).encode()).hexdigest()
                step(family, NAMES[family], item_id, "OK",
                     f"{table}: {rows:,} rows; {stored:,} {metric['sizeMetric']}; target {_SEED_TARGET_BYTES:,} bytes")
                return
        # Row widths are merely a first-batch proposal. Only the subsequent persisted metric decides success.
        bytes_per_row = stored / rows if rows else 2048
        increment = min(_SEED_TARGET_BYTES - stored, _SEED_TARGET_BYTES / 32)
        wanted = max(1, min(_SEED_MAX_ROWS, int(increment * .98 / bytes_per_row)))
        source = _seed_source(len(groups), wanted, as_delta=family == "Lakehouse")
        append(source)
    raise SeedError(f"{family} did not reach its physical size target within {_SEED_MAX_BATCHES} batches. "
                    "Inspect compression/publication errors; rerun the same parameters to resume. No data was removed.")


def seed_large_lakehouse(lh_id):
    pool = _seed_pool()
    def groups():
        return _seed_lh_groups(pool, seed_delta_snapshot(lh_id, pool["tablePath"]))
    _seed_drive("Lakehouse", lh_id, pool["table"], groups,
                lambda rows: seed_measure_delta("Lakehouse", lh_id, pool["table"], pool["tablePath"],
                                                rows, _SEED_TARGET_BYTES),
                lambda source: None)  # _seed_source commits the owned Delta batch atomically


def _seed_require_lakehouse():
    if SEED_METRICS.get("Lakehouse", {}).get("status") != "OK":
        raise SeedError("Large Lakehouse seeding/measurement failed. Bulk consumers are blocked; fix it and rerun "
                        "the same PREFIX. No new source or destination batches were appended.")
    return _seed_pool()


def _seed_sql_run(conn, statement):
    """Same SDK EXEC/SELECT contract as sql_run, but stress writes must not hide a commit failure."""
    escaped = statement.replace("'", "''")
    conn.query(f"EXEC(N'{escaped}'); SELECT 1 AS done;")
    if hasattr(conn, "commit"):
        conn.commit()


def seed_large_warehouse(wh_id, conn):
    pool = _seed_require_lakehouse()
    short, table = f"{PREFIX}_stress", f"dbo.{PREFIX}_stress"
    columns = conn.query("SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH FROM INFORMATION_SCHEMA.COLUMNS "
                         f"WHERE TABLE_SCHEMA = 'dbo' AND TABLE_NAME = '{short}' ORDER BY ORDINAL_POSITION")
    exists = columns is not None and len(columns) > 0
    _seed_destination_marker("Warehouse", wh_id, table, exists)
    if not exists:
        _seed_sql_run(conn, f"CREATE TABLE {table} (seed_owner VARCHAR(64) NOT NULL, seed_batch BIGINT NOT NULL, "
                            "seed_row BIGINT NOT NULL, payload VARCHAR(2048) NOT NULL)")
        columns = conn.query("SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH FROM INFORMATION_SCHEMA.COLUMNS "
                             f"WHERE TABLE_SCHEMA = 'dbo' AND TABLE_NAME = '{short}' ORDER BY ORDINAL_POSITION")
    expected = [("seed_owner", "varchar", 64), ("seed_batch", "bigint", None),
                ("seed_row", "bigint", None), ("payload", "varchar", 2048)]
    if columns is None or len(columns) != len(expected):
        raise SeedError("Warehouse stress schema missing or changed; no bulk load is safe.")
    for actual, (column, dtype, length) in zip(columns.to_dict("records"), expected):
        if (actual["COLUMN_NAME"] != column or str(actual["DATA_TYPE"]).lower() != dtype
                or (length is not None and int(actual["CHARACTER_MAXIMUM_LENGTH"]) != length)):
            raise SeedError("Warehouse stress schema collision; don't alter/adopt the existing table.")
    def groups():
        frame = conn.query(
            f"SELECT seed_batch, COUNT_BIG(*) AS [rows], COUNT_BIG(DISTINCT seed_row) AS distinct_rows, "
            "MIN(seed_row) AS first_row, MAX(seed_row) AS last_row, MIN(seed_owner) AS owner_min, "
            "MAX(seed_owner) AS owner_max, "
            f"SUM(CAST(CASE WHEN payload IS NULL OR LEN(payload) <> 2048 THEN 1 ELSE 0 END AS BIGINT)) AS bad_payload "
            f"FROM {table} GROUP BY seed_batch")
        return _seed_validate_groups([] if frame is None else frame.to_dict("records"))
    def append(source):
        # COPY accepts exact OneLake files using the executing notebook user's identity. No credentials/roles are created.
        locations = ", ".join("'" + _seed_url(pool["lh"], p) + "'" for p, _ in source["paths"])
        _seed_sql_run(conn, f"COPY INTO {table} FROM {locations} WITH (FILE_TYPE = 'PARQUET')")
    def verify(rows):
        frame = conn.query(f"SELECT seed_row, CONVERT(VARCHAR(64), HASHBYTES('SHA2_256', payload), 2) AS payload_hash "
                           f"FROM {table} WHERE seed_row IN (0, {rows - 1})")
        _seed_verify_hashes([] if frame is None else frame.to_dict("records"), rows)
    _seed_drive("Warehouse", wh_id, table, groups,
                lambda rows: seed_measure_delta("Warehouse", wh_id, table, f"Tables/dbo/{short}", rows, _SEED_TARGET_BYTES),
                append, verify)


def seed_large_eventhouse(eh_id, kql_id, uri, database):
    pool = _seed_require_lakehouse()
    table = f"{PREFIX}_stress"
    found = kusto(uri, database, f".show tables | where TableName == '{table}'")
    if len(found) > 1 or (found and found[0].get("DocString") != pool["description"]):
        raise SeedError("Eventhouse stress ownership collision; don't merge/alter an existing table.")
    _seed_destination_marker("Eventhouse", kql_id, table, bool(found))
    if not found:
        definition = ", ".join(f"{c}:{t}" for c, t in _SEED_COLUMNS)
        kusto(uri, database, f".create table {table} ({definition}) with (docstring='{pool['description']}')")
    schema = kusto(uri, database, f".show table {table} schema as json")
    actual = json.loads(schema[0]["Schema"]).get("OrderedColumns", []) if len(schema) == 1 else []
    if [(c.get("Name"), c.get("CslType")) for c in actual] != _SEED_COLUMNS:
        raise SeedError("Eventhouse stress schema collision; no ingestion is safe.")
    def groups():
        records = kusto(uri, database,
                        f"{table} | summarize rows=count(), distinct_rows=count_distinct(seed_row), "
                        "first_row=min(seed_row), last_row=max(seed_row), owner_min=take_any(seed_owner), "
                        f"owner_max=take_any(seed_owner), bad_payload=countif(seed_owner != '{pool['owner']}' "
                        "or isempty(payload) or strlen(payload) != 2048) "
                        "by seed_batch", "query")
        result = _seed_validate_groups(records)
        if result:
            allowed = {"ingest-by:" + _seed_tag(pool, n) for n in result}
            tagsets = kusto(uri, database, f"{table} | extend ExtentTags=extent_tags() "
                            "| summarize by Tags=tostring(ExtentTags)", "query")
            if not tagsets:
                raise SeedError("No extent-tag evidence returned for populated KQL data; no retry is safe.")
            for extent in tagsets:
                tags = json.loads(extent["Tags"]) if extent.get("Tags") else []
                if (not isinstance(tags, list) or not tags or not all(isinstance(t, str) for t in tags)
                        or not set(tags).issubset(allowed)):
                    raise SeedError("Untracked KQL extent tags in the stress table. Refusing retry/ownership ambiguity.")
        return result
    def append(source):
        locators = ", ".join(f"h'abfss://{WS}@{ONELAKE_DFS.split('://', 1)[1]}/{pool['lh']}/"
                              f"{quote(_seed_relative(p), safe='/')};impersonate'" for p, _ in source["paths"])
        tag = _seed_tag(pool, source["seed_batch"])
        mapping = json.dumps([{"Column": c, "DataType": t, "Properties": {"Field": c}} for c, t in _SEED_COLUMNS],
                             separators=(",", ":"))
        results = kusto(uri, database, f".ingest into table {table} ({locators}) with (format='parquet', "
                        f"ingestionMapping='{mapping}', ingestIfNotExists='{json.dumps([tag])}', "
                        f"tags='{json.dumps(['ingest-by:' + tag])}')")
        if not results or any(r.get("HasErrors") not in (False, "false", "False") for r in results):
            raise SeedError("KQL bulk ingestion failed or omitted its result: " + json.dumps(results) +
                            ". Inspect ingestion errors; rerun only the same owned batches, never remove tags.")
    def verify(rows):
        records = kusto(uri, database, f"{table} | where seed_row in (0, {rows - 1}) "
                        "| project seed_row, payload_hash=hash_sha256(payload)", "query")
        _seed_verify_hashes(records, rows)
    _seed_drive("Eventhouse", eh_id, table, groups,
                lambda rows: seed_measure_kql(eh_id, uri, database, table, rows, _SEED_TARGET_BYTES), append, verify)


def _seed_tag(pool, batch):
    return f"v2t-{pool['owner']}-{batch:06d}"

## 1. Lakehouse, its data, a shortcut, and a view on its SQL analytics endpoint

In [ ]:
def create_lakehouse():
    t, name = "Lakehouse", NAMES["Lakehouse"]
    lh_id, status = ensure_item(t, name)
    IDS[t] = lh_id
    step(t, name, lh_id, status)

    # Files: a CSV and a nested text file
    rows = ["id,product,amount"] + [f"{i},product-{i % 7},{i * 1.5:.2f}" for i in range(1, 101)]
    onelake_write(lh_id, "Files/raw/sample.csv", "\n".join(rows) + "\n")
    onelake_write(lh_id, "Files/raw/nested/readme.txt", "Test file for the Fabric migration notebooks.\n")
    step(t, name, lh_id, "OK", "Files/raw/sample.csv and Files/raw/nested/readme.txt written")

    # Delta table (deltalake is preinstalled in Fabric Python notebooks)
    try:
        from deltalake import write_deltalake
        df = pd.DataFrame({"id": range(1, 1001), "region": [["North", "South", "East", "West"][i % 4] for i in range(1000)],
                           "amount": [round(i * 2.5, 2) for i in range(1000)],
                           "sale_date": pd.date_range("2026-01-01", periods=1000, freq="h")})
        if LARGE_DATA_MIB or MEASURE_DATA_SIZE:
            small_path = seed_small_lakehouse(lh_id, df)
        else:
            write_deltalake(f"abfss://{WS}@onelake.dfs.fabric.microsoft.com/{lh_id}/Tables/{PREFIX}_sales", df,
                            mode="overwrite",
                            storage_options={"bearer_token": notebookutils.credentials.getToken("storage"),
                                             "use_fabric_endpoint": "true"})
        step(t, name, lh_id, "OK", f"Delta table {PREFIX}_sales: 1,000 rows")
    except Exception as exc:
        seed_fail(t, exc)
        step(t, name, lh_id, "Failed", f"Delta table: {exc}")
        if LARGE_DATA_MIB or MEASURE_DATA_SIZE:
            raise

    # Shortcut to the Lakehouse's own Files/raw folder: the backup must skip shortcut data
    try:
        fabric("POST", f"/workspaces/{WS}/items/{lh_id}/shortcuts?shortcutConflictPolicy=Abort",
               {"path": "Files", "name": f"{PREFIX}_shortcut",
                "target": {"oneLake": {"workspaceId": WS, "itemId": lh_id, "path": "Files/raw"}}})
        step(t, name, lh_id, "OK", f"shortcut Files/{PREFIX}_shortcut -> Files/raw")
    except HttpError as exc:
        if exc.status == 409 or "exist" in exc.message.lower():
            step(t, name, lh_id, "Exists", f"shortcut Files/{PREFIX}_shortcut")
        else:
            step(t, name, lh_id, "Failed", f"shortcut: {exc}")

    # SQL analytics endpoint: wait until provisioned, then add a view
    t2 = "SQLEndpoint"
    try:
        def endpoint():
            props = fabric("GET", f"/workspaces/{WS}/lakehouses/{lh_id}").get("properties") or {}
            sql = props.get("sqlEndpointProperties") or {}
            return sql if sql.get("id") and str(sql.get("provisioningStatus", "Success")).lower() == "success" else None
        sql = wait_until(endpoint, "The SQL analytics endpoint")
        IDS[t2] = sql["id"]
        conn = sql_connect(lh_id, "Lakehouse")
        sql_run(conn, f"CREATE OR ALTER VIEW dbo.{PREFIX}_view AS SELECT 1 AS id, 'backup test' AS note")
        ok = sql_scalar(conn, f"SELECT COUNT(*) FROM sys.views WHERE name = '{PREFIX}_view'")
        step(t2, name, sql["id"], "OK" if ok else "Failed", f"view dbo.{PREFIX}_view" + ("" if ok else " not found after creation"))
    except Exception as exc:
        step(t2, name, IDS.get(t2), "Failed", str(exc))

    if LARGE_DATA_MIB:
        seed_large_lakehouse(lh_id)
    elif MEASURE_DATA_SIZE:
        if "small_path" not in locals():
            raise SeedError("The original small Lakehouse fixture failed; persisted-size measurement is blocked.")
        seed_measure_delta(t, lh_id, f"{PREFIX}_sales", small_path, 1000)


if "Lakehouse" in CREATE_TYPES:
    try:
        create_lakehouse()
    except Exception as exc:
        seed_fail("Lakehouse", exc)
        step("Lakehouse", NAMES["Lakehouse"], IDS.get("Lakehouse"), "Failed", str(exc))

## 2. Warehouse with a table, a view and a stored procedure

In [ ]:
def create_warehouse():
    t, name = "Warehouse", NAMES["Warehouse"]
    wh_id, status = ensure_item(t, name)
    IDS[t] = wh_id
    step(t, name, wh_id, status)
    conn = wait_until(lambda: _try_connect(wh_id), "The Warehouse", every=15)
    table = f"dbo.{PREFIX}_orders"
    sql_run(conn, f"IF OBJECT_ID('{table}') IS NULL CREATE TABLE {table} "
                  "(OrderId INT, Customer VARCHAR(50), Amount DECIMAL(10,2), OrderDate DATE)")
    if int(sql_scalar(conn, f"SELECT COUNT(*) FROM {table}") or 0) == 0:
        values = ",".join(f"({i}, 'customer-{i % 25}', {i * 3.75:.2f}, DATEADD(day, {i % 365}, '2026-01-01'))"
                          for i in range(1, 501))
        sql_run(conn, f"INSERT INTO {table} (OrderId, Customer, Amount, OrderDate) VALUES {values}")
    sql_run(conn, f"CREATE OR ALTER VIEW dbo.{PREFIX}_orders_by_customer AS "
                  f"SELECT Customer, COUNT(*) AS Orders, SUM(Amount) AS Total FROM {table} GROUP BY Customer")
    sql_run(conn, f"CREATE OR ALTER PROCEDURE dbo.{PREFIX}_top_customers AS "
                  f"SELECT TOP 5 * FROM dbo.{PREFIX}_orders_by_customer ORDER BY Total DESC")
    rows = int(sql_scalar(conn, f"SELECT COUNT(*) FROM {table}") or 0)
    step(t, name, wh_id, "OK" if rows == 500 else "Check", f"{table}: {rows} rows; view and procedure created")
    if LARGE_DATA_MIB or MEASURE_DATA_SIZE:
        if rows != 500:
            raise SeedError(f"Original small Warehouse fixture has {rows} rows, expected 500; no stress load is safe.")
        if LARGE_DATA_MIB:
            seed_large_warehouse(wh_id, conn)
        else:
            seed_measure_delta(t, wh_id, table, f"Tables/dbo/{PREFIX}_orders", rows)


def _try_connect(wh_id):
    try:
        conn = sql_connect(wh_id, "Warehouse")
        conn.query("SELECT 1 AS ready")
        return conn
    except RuntimeError:
        raise
    except Exception:
        return None


if "Warehouse" in CREATE_TYPES:
    try:
        create_warehouse()
    except Exception as exc:
        seed_fail("Warehouse", exc)
        step("Warehouse", NAMES["Warehouse"], IDS.get("Warehouse"), "Failed", str(exc))

## 3. Eventhouse with a KQL table and a function
Fabric creates a KQL database with the same name inside the Eventhouse.

In [ ]:
def create_eventhouse():
    t, name = "Eventhouse", NAMES["Eventhouse"]
    eh_id, status = ensure_item(t, name)
    IDS[t] = eh_id
    step(t, name, eh_id, status)

    def database():
        for db in fabric_list(f"/workspaces/{WS}/kqlDatabases"):
            props = db.get("properties") or {}
            if (props.get("parentEventhouseItemId") or "").lower() == eh_id.lower() and props.get("queryServiceUri"):
                return db
        return None
    db = wait_until(database, "The Eventhouse's KQL database")
    IDS["KQLDatabase"] = db["id"]
    uri, dbname = db["properties"]["queryServiceUri"], db["properties"].get("databaseName") or db["displayName"]
    table = f"{PREFIX}_events"
    kusto(uri, dbname, f".create-merge table {table} (Timestamp:datetime, DeviceId:string, Value:real)")
    count = int((kusto(uri, dbname, f"{table} | count", "query") or [{"Count": 0}])[0].get("Count") or 0)
    if count == 0:
        kusto(uri, dbname, f".set-or-append {table} <| range i from 1 to 1000 step 1 "
                           "| project Timestamp = datetime(2026-01-01) + i * 1m, DeviceId = strcat('device-', i % 10), "
                           "Value = todouble(i % 100) / 3.0")
        count = int(kusto(uri, dbname, f"{table} | count", "query")[0].get("Count") or 0)
    kusto(uri, dbname, f".create-or-alter function with (folder = 'backup-test') {PREFIX}_avg_by_device() "
                       f"{{ {table} | summarize AvgValue = avg(Value) by DeviceId }}")
    step("KQLDatabase", db["displayName"], db["id"], "OK" if count == 1000 else "Check",
         f"table {table}: {count:,} rows; function {PREFIX}_avg_by_device")
    if LARGE_DATA_MIB or MEASURE_DATA_SIZE:
        if count != 1000:
            raise SeedError(f"Original small KQL fixture has {count} rows, expected 1000; no stress load is safe.")
        if LARGE_DATA_MIB:
            seed_large_eventhouse(eh_id, db["id"], uri, dbname)
        else:
            seed_measure_kql(eh_id, uri, dbname, table, count)


if "Eventhouse" in CREATE_TYPES:
    try:
        create_eventhouse()
    except Exception as exc:
        seed_fail("Eventhouse", exc)
        step("Eventhouse", NAMES["Eventhouse"], IDS.get("Eventhouse"), "Failed", str(exc))

## 4. Notebook, pipeline, dataflow, data agent, graph model and ontology
The pipeline runs the test notebook, and the notebook's default Lakehouse is the test Lakehouse,
so the backup also contains references between items.

In [ ]:
def notebook_definition():
    meta = {"language_info": {"name": "python"},
            "kernel_info": {"name": "synapse_pyspark"},
            "microsoft": {"language": "python", "language_group": "synapse_pyspark"}}
    if IDS.get("Lakehouse"):
        meta["dependencies"] = {"lakehouse": {"default_lakehouse": IDS["Lakehouse"],
                                              "default_lakehouse_name": NAMES["Lakehouse"],
                                              "default_lakehouse_workspace_id": WS}}
    cells = [{"cell_type": "markdown", "metadata": {}, "source": ["# Backup test notebook\n", "Reads the test Delta table."]},
             {"cell_type": "code", "metadata": {}, "execution_count": None, "outputs": [],
              "source": [f'df = spark.read.table("{PREFIX}_sales")\n', "display(df.limit(10))"]}]
    nb = {"nbformat": 4, "nbformat_minor": 5, "cells": cells, "metadata": meta}
    return {"format": "ipynb", "parts": [{"path": "artifact.content.ipynb", "payload": b64(nb),
                                          "payloadType": "InlineBase64"}]}


def pipeline_definition():
    activities = [{"name": "Wait 5 seconds", "type": "Wait", "dependsOn": [], "typeProperties": {"waitTimeInSeconds": 5}}]
    if IDS.get("Notebook"):
        activities.append({"name": "Run test notebook", "type": "TridentNotebook",
                           "dependsOn": [{"activity": "Wait 5 seconds", "dependencyConditions": ["Succeeded"]}],
                           "policy": {"timeout": "0.01:00:00", "retry": 0},
                           "typeProperties": {"notebookId": IDS["Notebook"], "workspaceId": WS}})
    content = {"properties": {"description": "Test pipeline for the Fabric migration notebooks", "activities": activities}}
    return {"parts": [{"path": "pipeline-content.json", "payload": b64(content), "payloadType": "InlineBase64"}]}


def dataflow_definition():
    query_id = str(uuid.uuid4())
    metadata = {"formatVersion": "202502", "name": NAMES["Dataflow"], "documentLocale": "en-US", "queryGroups": [],
                "queriesMetadata": {"TestQuery": {"queryId": query_id, "queryName": "TestQuery", "loadEnabled": False}}}
    mashup = ('section Section1;\nshared TestQuery = let\n'
              '    Source = #table(type table [id = Int64.Type, name = text], {{1, "alpha"}, {2, "beta"}, {3, "gamma"}})\n'
              'in\n    Source;\n')
    return {"parts": [{"path": "queryMetadata.json", "payload": b64(metadata), "payloadType": "InlineBase64"},
                      {"path": "mashup.pq", "payload": b64(mashup), "payloadType": "InlineBase64"}]}


DEFINITIONS = {"Notebook": notebook_definition, "DataPipeline": pipeline_definition, "Dataflow": dataflow_definition}

for item_type in ["Notebook", "DataPipeline", "Dataflow", "DataAgent", "GraphModel", "Ontology"]:
    if item_type not in CREATE_TYPES:
        continue
    name = NAMES[item_type]
    make = DEFINITIONS.get(item_type)
    try:
        try:
            item_id, status = ensure_item(item_type, name, make() if make else None)
            detail = "with definition" if make and status == "Created" else ""
        except HttpError as exc:
            if not make or exc.status in (401, 403):
                raise
            item_id, status = ensure_item(item_type, name)   # the definition was rejected: create it empty
            detail = f"created empty; the definition was rejected: {exc}"
        IDS[item_type] = item_id
        step(item_type, name, item_id, status, detail)
    except HttpError as exc:
        hint = " (this item type may be in preview or turned off in the tenant settings)" \
            if exc.status == 400 or "type" in exc.code.lower() else ""
        step(item_type, name, None, "Failed", f"{exc}{hint}")

## 5. Result

In [ ]:
result = pd.DataFrame(STEPS)
try:
    display(result)
except NameError:
    print(result.to_string())
failed = result[result.Status.isin(["Failed", "Check"])]
print(f"\n{len(failed)} step(s) need attention." if len(failed) else "\nAll steps succeeded.")
print("\nPaste this into the settings cell of V2_0 to V2_3 (V2_4 to V2_6 take the workspaces from the snapshot):")
print(f'WORKSPACE_IDS = ["{WS}"]')
if LARGE_DATA_MIB or MEASURE_DATA_SIZE:
    print("\nSEED_METRICS (physical bytes, no logical-size estimates):")
    print(json.dumps(SEED_METRICS, indent=2, sort_keys=True))
    incomplete = {k: v for k, v in SEED_METRICS.items() if v["status"] != "OK"}
    if incomplete or len(failed):
        raise SeedError("Test-item seeding/measurement is incomplete. "
                        + json.dumps(incomplete, sort_keys=True) + "\n"
                        + "\n".join(str(s["Detail"]) for s in STEPS if s["Status"] in ("Failed", "Check"))
                        + "\nFix the reported publishing/ingestion/ownership error and rerun the same parameters. "
                          "No size waiver, cleanup, permission or policy change was performed.")

## 6. Delete the test items (optional, required before the migration)
Hard-deletes **every item in the target workspace whose name starts with the prefix** (`bkp_test_` by default) and,
with `DELETE_DATAFLOW_STAGING = True`, the Dataflow Gen2 staging items Fabric created in this workspace. It doesn't
run unless you set `DELETE_TEST_ITEMS = True`. With `HARD_DELETE = True` (default) the items are gone for good and
don't sit in the recycle bin, where they would block the tenant migration. Hard delete needs the workspace Admin
role.

In [ ]:
DELETE_TEST_ITEMS = False
HARD_DELETE = True               # False = soft delete (recoverable, but a soft-deleted item blocks the migration)
DELETE_DATAFLOW_STAGING = True   # also StagingLakehouseForDataflows_* / StagingWarehouseForDataflows_* of this workspace

STAGING = re.compile(r"^Staging(Lakehouse|Warehouse)ForDataflows(_\d+)?$", re.IGNORECASE)
if DELETE_TEST_ITEMS:
    # SQL endpoints and KQL databases go with their Lakehouse and Eventhouse; pipelines first, they reference notebooks.
    order = ["DataPipeline", "Notebook", "Dataflow", "DataAgent", "GraphModel", "Ontology", "Warehouse",
             "Lakehouse", "Eventhouse"]
    items = [i for i in fabric_list(f"/workspaces/{WS}/items") if i.get("type") in order and (
        i.get("displayName", "").lower().startswith(PREFIX.lower() + "_")
        or (DELETE_DATAFLOW_STAGING and STAGING.match(i.get("displayName", ""))))]
    mode = "hard" if HARD_DELETE else "soft"
    for item in sorted(items, key=lambda i: order.index(i["type"])):
        try:
            fabric("DELETE", f"/workspaces/{WS}/items/{item['id']}" + ("?hardDelete=true" if HARD_DELETE else ""))
            print(f"  deleted ({mode}) {item['type']} {item['displayName']}")
        except HttpError as exc:
            print(f"  ! {item['type']} {item['displayName']}: {exc}")
    print(f"{len(items)} test item(s) processed ({mode} delete).")
else:
    print("Nothing deleted. Set DELETE_TEST_ITEMS = True in this cell and run it to remove the test items.")